In [ ]:
import os, cv2, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNet
from sklearn.model_selection import StratifiedKFold
import keras_tuner as kt

DATASET_DIR = "/kaggle/input/mangoleafds224x224/MangoLeafDS224x224"
IMG_SIZE, BATCH_SIZE, EPOCHS = 224, 32, 10
N_SPLITS, NUM_CLASSES, SEED = 5, 6, 42
tf.random.set_seed(SEED); np.random.seed(SEED)

paths, labels = [], []
for i, c in enumerate(sorted(os.listdir(DATASET_DIR))):
    for f in os.listdir(os.path.join(DATASET_DIR, c)):
        paths.append(os.path.join(DATASET_DIR, c, f)); labels.append(i)
paths, labels = np.array(paths), np.array(labels)

def gen(p, l):
    while True:
        idx = np.random.permutation(len(p))
        for i in range(0, len(p), BATCH_SIZE):
            imgs, labs = [], []
            for j in idx[i:i+BATCH_SIZE]:
                im = cv2.resize(cv2.cvtColor(cv2.imread(p[j]), cv2.COLOR_BGR2RGB),(IMG_SIZE,IMG_SIZE))
                imgs.append(im); labs.append(l[j])
            yield np.array(imgs)/255., tf.keras.utils.to_categorical(labs, NUM_CLASSES)

def model_builder(hp):
    base = MobileNet(weights="imagenet", include_top=False, input_shape=(224,224,3))
    base.trainable = False
    x = layers.Flatten()(base.output)
    x = layers.Dense(hp.Choice("units",[1024,2048]),activation="relu")(x)
    x = layers.Dropout(hp.Float("drop",0.3,0.6,0.1))(x)
    out = layers.Dense(NUM_CLASSES,activation="softmax")(x)
    m = models.Model(base.input,out)
    m.compile(optimizer=tf.keras.optimizers.Adam(hp.Choice("lr",[1e-3,1e-4])),
              loss="categorical_crossentropy",metrics=["accuracy"])
    return m

skf = StratifiedKFold(5,shuffle=True,random_state=SEED)
for f,(tr,va) in enumerate(skf.split(paths,labels),1):
    tuner = kt.BayesianOptimization(model_builder,"val_accuracy",max_trials=8,
                                    directory="MBV1",project_name=f"fold{f}")
    tuner.search(gen(paths[tr],labels[tr]),
                 steps_per_epoch=len(tr)//BATCH_SIZE,
                 validation_data=gen(paths[va],labels[va]),
                 validation_steps=len(va)//BATCH_SIZE,
                 epochs=EPOCHS)
